In [1]:
import pandas as pd
import numpy as np

def impute_missing_data(df, strategy='median'):
    df_clean = df.copy()
    if strategy == 'median':
        return df_clean.fillna(df_clean.median(numeric_only=True))
    else:
        return df_clean.fillna(df_clean.mean(numeric_only=True))

def encode_categorical(df, one_hot_cols=None, ordinal_mapping=None):
    df_encoded = df.copy()
    if ordinal_mapping:
        for col, mapping in ordinal_mapping.items():
            df_encoded[col] = df_encoded[col].map(mapping)
    if one_hot_cols:
        df_encoded = pd.get_dummies(df_encoded, columns=one_hot_cols, drop_first=True)
    return df_encoded

def scale_features(df, cols, method='zscore'):
    df_scaled = df.copy()
    for col in cols:
        if method == 'zscore':
            df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()
        elif method == 'minmax':
            min_val = df_scaled[col].min()
            max_val = df_scaled[col].max()
            df_scaled[col] = (df_scaled[col] - min_val) / (max_val - min_val)
    return df_scaled

def calculate_cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_v1 = np.linalg.norm(vec1)
    norm_v2 = np.linalg.norm(vec2)
    return dot_product / (norm_v1 * norm_v2)

def main():
    print("--- Running A1 Preprocessing & Similarity ---")
    df = pd.DataFrame({'Age': [20, np.nan, 40, 30], 'Score': [100, 200, np.nan, 400]})
    df_clean = impute_missing_data(df)
    print("Imputed Data:\n", df_clean)
    
    df_scaled = scale_features(df_clean, ['Age', 'Score'])
    print("\nScaled Data:\n", df_scaled)
    
    sim = calculate_cosine_similarity(np.array([1, 2]), np.array([1, 2]))
    print("\nCosine Similarity:", sim)

if __name__ == '__main__':
    main()

--- Running A1 Preprocessing & Similarity ---
Imputed Data:
     Age  Score
0  20.0  100.0
1  30.0  200.0
2  40.0  200.0
3  30.0  400.0

Scaled Data:
         Age     Score
0 -1.224745 -0.993399
1  0.000000 -0.198680
2  1.224745 -0.198680
3  0.000000  1.390759

Cosine Similarity: 0.9999999999999998


In [2]:
import unittest
import pandas as pd
import numpy as np

def impute_missing_data(df):
    return df.fillna(df.median(numeric_only=True))

def scale_features(df, cols):
    df_scaled = df.copy()
    for col in cols:
        df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()
    return df_scaled

def calculate_cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def custom_kmeans(X, k=2):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    for _ in range(50):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
    return labels, centroids

class TestLabFunctions(unittest.TestCase):

    def setUp(self):
        self.test_df = pd.DataFrame({'Age': [20, np.nan, 40, 30]})
        self.X_data = np.array([[1, 2], [1, 4], [10, 2], [10, 4]])

    def test_impute_missing_data(self):
        cleaned_df = impute_missing_data(self.test_df)
        self.assertEqual(cleaned_df['Age'].isnull().sum(), 0)

    def test_scale_features(self):
        scaled_df = scale_features(self.test_df.fillna(0), ['Age'])
        self.assertAlmostEqual(scaled_df['Age'].mean(), 0.0, places=4)

    def test_cosine_similarity(self):
        v1 = np.array([1, 0])
        v2 = np.array([1, 0])
        self.assertAlmostEqual(calculate_cosine_similarity(v1, v2), 1.0)

    def test_custom_kmeans(self):
        labels, centroids = custom_kmeans(self.X_data, k=2)
        self.assertEqual(len(labels), len(self.X_data))
        self.assertEqual(len(centroids), 2)

def main():
    print("--- Running A2 Unit Tests ---")
    unittest.main(argv=[''], exit=False)

if __name__ == '__main__':
    main()

....

--- Running A2 Unit Tests ---



----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


In [4]:
import numpy as np
import time
from sklearn.cluster import KMeans

def custom_kmeans(X, k=3, max_iters=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    
    for _ in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
        
    return labels, centroids

def ai_kmeans(X, k=3):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    return labels, model.cluster_centers_

def compare_kmeans_performance(X, k=3):
    start_custom = time.time()
    custom_labels, _ = custom_kmeans(X, k)
    time_custom = time.time() - start_custom
    
    start_ai = time.time()
    ai_labels, _ = ai_kmeans(X, k)
    time_ai = time.time() - start_ai
    
    print("Custom K-Means Execution Time:", time_custom, "seconds")
    print("AI (Sklearn) Execution Time:", time_ai, "seconds")

def main():
    print("--- Running A3 K-Means Performance Comparison ---")
    dataset = np.random.rand(500, 4)
    compare_kmeans_performance(dataset, k=3)

if __name__ == '__main__':
    main()

--- Running A3 K-Means Performance Comparison ---
Custom K-Means Execution Time: 0.0008318424224853516 seconds
AI (Sklearn) Execution Time: 0.010361194610595703 seconds
